# Histogram

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SCI_PATH = Path("pr_metrics_all_sci.csv")
NON_PATH = Path("pr_metrics_all_non_sci.csv")
OUT_DIR  = Path("histograms_log_pages")

METRICS = [
    "Time to Merge (days)",
    "Number of Unique Reviewers",
    "Total Discussion Comments",
    "Commits After First Review",
    "Time Between First and Last Comments (days)",
    "Requested Changes",
]
REPO_COL   = "repo"
LOG_PREFIX = "log1p"

# HELPERS
def ensure_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

def get_log_values(df: pd.DataFrame, metric: str) -> pd.Series:
    """
    Return log1p-transformed values for a metric.
    Prefer an existing CSV column named 'log1p <metric>'.
    If it's missing, compute np.log1p(metric) safely.
    """
    col_log = f"{LOG_PREFIX}{metric}"
    if col_log in df.columns:
        return ensure_numeric(df[col_log])
    # fallback: compute log1p from raw column
    if metric not in df.columns:
        raise ValueError(f"Neither '{col_log}' nor raw '{metric}' found in DataFrame.")
    raw = ensure_numeric(df[metric])
    raw = raw.where(raw >= 0)  # if negatives -> NaN
    return np.log1p(raw)

def freedman_diaconis_bins(x: np.ndarray, min_bins: int = 10, max_bins: int = 50) -> int:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    n = x.size
    if n < 2:
        return min_bins
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    if iqr == 0:
        bins = int(np.sqrt(n))
    else:
        h = 2 * iqr * (n ** (-1/3))
        if h <= 0:
            bins = int(np.sqrt(n))
        else:
            rng = x.max() - x.min()
            bins = int(np.ceil(rng / h)) if rng > 0 else min_bins
    return max(min_bins, min(max_bins, max(1, bins)))

def common_edges_across_all(df: pd.DataFrame, metric: str) -> np.ndarray:
    """
    Build common bin edges on the log scale using ALL repos and BOTH labels.
    """
    vals = get_log_values(df, metric).to_numpy(dtype=float)
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
        return np.array([0.0, 1.0], dtype=float)
    nbins = freedman_diaconis_bins(vals)
    edges = np.histogram_bin_edges(vals, bins=nbins)
    edges = np.unique(edges)
    if edges.size < 2:
        lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
        if lo == hi: hi = lo + 1.0
        edges = np.array([lo, hi], dtype=float)
    return edges

def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    sci = pd.read_csv(SCI_PATH)
    non = pd.read_csv(NON_PATH)
    sci["__label__"] = "Scientific"
    non["__label__"] = "Non-Scientific"
    df = pd.concat([sci, non], ignore_index=True)

    if REPO_COL not in df.columns:
        raise ValueError(f"Missing repo column: {REPO_COL}")

    repos = sorted(df[REPO_COL].dropna().astype(str).unique())
    if not repos:
        raise SystemExit("No repositories found.")

    # One figure per metric, 3 subplots (one per repo), all sharing x/y
    for metric in METRICS:
        # Precompute shared bins on log scale
        edges = common_edges_across_all(df, metric)
        centers   = (edges[:-1] + edges[1:]) / 2.0
        bin_width = (edges[1:] - edges[:-1])
        bar_w     = bin_width * 0.4

        # Cache counts and determine global y-max
        ymax_all = 0
        counts_cache = {}
        for repo in repos:
            d_r = df[df[REPO_COL].astype(str) == repo]
            sci_vals_log = get_log_values(d_r[d_r["__label__"] == "Scientific"], metric).to_numpy(dtype=float)
            non_vals_log = get_log_values(d_r[d_r["__label__"] == "Non-Scientific"], metric).to_numpy(dtype=float)

            sci_counts, _ = np.histogram(sci_vals_log[~np.isnan(sci_vals_log)], bins=edges)
            non_counts, _ = np.histogram(non_vals_log[~np.isnan(non_vals_log)], bins=edges)

            counts_cache[repo] = (sci_counts, non_counts)
            ymax_all = max(ymax_all, sci_counts.max(initial=0), non_counts.max(initial=0))

        # Create subplots
        n_repos = len(repos)
        fig, axes = plt.subplots(
            nrows=1,
            ncols=n_repos,
            figsize=(5.0 * n_repos, 4.2),
            sharex=True,
            sharey=True,
        )
        if n_repos == 1:
            axes = [axes]

        # Plot
        for ax, repo in zip(axes, repos):
            sci_counts, non_counts = counts_cache[repo]
            ax.bar(centers - bar_w/2, sci_counts, width=bar_w, alpha=0.85, label="Scientific", align="center")
            ax.bar(centers + bar_w/2, non_counts, width=bar_w, alpha=0.85, label="Non-Scientific", align="center")

            ax.set_title(repo)
            ax.set_xlim(edges[0], edges[-1])
            ax.set_ylim(0, ymax_all * 1.08 if ymax_all > 0 else 1)
            # Axis label
            ax.set_xlabel(f"log(1 + {metric})")

            # Clear subplot title
            ax.set_title(f"{repo} — {metric}")


        axes[0].set_ylabel("Number of PRs")

        # Single legend at top-center
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.05))

        fig.suptitle(f"Histograms — {metric} (log scale)", y=1.12, fontsize=12)
        fig.tight_layout()

        safe_metric = metric.replace("/", "_").replace(" ", "_")
        out_png = OUT_DIR / f"{safe_metric}__all_repos_log.png"
        plt.savefig(out_png, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"[OK] Saved {out_png}")

if __name__ == "__main__":
    main()

[OK] Saved histograms_log_pages/Time_to_Merge_(days)__all_repos_log.png
[OK] Saved histograms_log_pages/Number_of_Unique_Reviewers__all_repos_log.png
[OK] Saved histograms_log_pages/Total_Discussion_Comments__all_repos_log.png
[OK] Saved histograms_log_pages/Commits_After_First_Review__all_repos_log.png
[OK] Saved histograms_log_pages/Time_Between_First_and_Last_Comments_(days)__all_repos_log.png
[OK] Saved histograms_log_pages/Requested_Changes__all_repos_log.png
